# 실습 5: 센서 590개 진단표 만들기
- 상황: 관리도로 한 개는 봤는데 나머지 589개가 남았다
- 목표: 열 하나를 한 줄로 요약해 쓸 수 없는 열을 걸러낸다

## Step 0. 폴더와 노트북 만들고 데이터 불러오기

In [1]:
import pandas as pd

df = pd.read_csv("../../data/04_secom.csv")
print("shape (행, 열):", df.shape)


shape (행, 열): (1567, 592)


## Step 1. 진단 항목 정하기

### 용어 풀이 - 열을 진단할 때 쓰는 말

| 말 | 뜻 |
|---|---|
| 진단표 | 열 하나가 한 줄이 되도록 요약한 표. 590개 열이 590줄이 된다 |
| 빈칸 비율 (결측률) | 그 열에서 값이 비어 있는 칸의 비율 |
| 값 종류 수 | 그 열에 서로 다른 값이 몇 가지 들어 있는지 |
| 상수열 | 값 종류가 1개뿐인 열. 처음부터 끝까지 같은 값만 나온다 |
| 스케일 | 값의 크기 단위. 어떤 열은 0~1, 어떤 열은 수천이라 그대로 비교하면 안 된다 |
| 임계값 | 버릴지 말지를 가르는 경계 숫자. 정답이 없어 사람이 정한다 |
| 1차 선별 | 쓸 수 없는 열을 먼저 떨어내는 단계. 쓸모를 따지는 건 그다음이다 |

[내 진단표에 넣을 항목]<br>
1. 빈칸 비율   2. 값 종류 수   3. 표준편차   4. 최소와 최대

## Step 2. 열 하나로 먼저 직접 계산해보기

### 열 하나로 진단 네 값 구해보기
590개에 시키기 전에, 한 열로 계산 방법을 익힌다.

In [2]:
# 앞 실습에서 골랐던 센서 이름을 다시 넣는다 (아무 센서나 괜찮습니다)
센서 = "sensor_089"

# isna() — 빈칸이면 참(True), 아니면 거짓(False)
# mean() — 참/거짓의 평균은 곧 참의 비율이 된다 (참=1, 거짓=0이라서)
빈칸비율 = df[센서].isna().mean() * 100

# nunique() — 서로 다른 값이 몇 종류인지 센다
값종류수 = df[센서].nunique()

# std() — 값이 평균에서 얼마나 흩어져 있는지
표준편차 = df[센서].std()

print("열 이름:", 센서)
print("빈칸 비율:", round(빈칸비율, 2), "%")
print("값 종류 수:", 값종류수)
print("표준편차:", round(표준편차, 4))
print("최소~최대:", df[센서].min(), "~", df[센서].max())

열 이름: sensor_089
빈칸 비율: 0.0 %
값 종류 수: 973
표준편차: 53.5373
최소~최대: 1627.4714 ~ 2105.1823


### 문법 노트 - 열 하나를 숫자로 줄이기

| 쓴 것 | 하는 일 | 왜 여기 쓰나 |
|---|---|---|
| .isna() | 빈칸이면 참, 아니면 거짓 | 빈칸을 세려면 먼저 참/거짓으로 바꿔야 한다 |
| .mean() | 평균 | 참/거짓에 쓰면 참의 비율이 된다 |
| .nunique() | 서로 다른 값의 개수 | 1이면 상수열. 이걸로 바로 판정된다 |
| .std() | 표준편차 | 0에 가까우면 거의 안 변하는 열 |
| .min() .max() | 가장 작은 값 / 가장 큰 값 | 값의 범위. 스케일 감을 잡는다 |

**.isna().mean()이 왜 비율이 되나?**<br>
참을 1, 거짓을 0으로 놓고 평균을 내기 때문이다.<br>
100개 중 4개가 빈칸이면 (1+1+1+1+0+0+...)/100 = 0.04 이므로 4%.<br>
개수를 세고 전체로 나누는 두 단계가 한 줄에 들어간 셈이다.<br>
앞으로 "비율을 구한다" 하면 거의 이 형태가 나온다.

## Step 3. 590개 전체는 AI에게 시키기

In [3]:
sensor_cols = [c for c in df.columns if c.startswith("sensor_")]

rows = []
for c in sensor_cols:
    rows.append({
        "열 이름": c,
        "빈칸 비율(%)": round(df[c].isna().mean() * 100, 2),
        "값 종류 수": df[c].nunique(),
        "표준편차": df[c].std(),
        "최솟값": df[c].min(),
        "최댓값": df[c].max(),
    })

sensor_diagnosis = pd.DataFrame(rows)

print("sensor_diagnosis 전체 줄 수:", len(sensor_diagnosis))

sensor_diagnosis.sort_values("빈칸 비율(%)", ascending=False).head(10)


sensor_diagnosis 전체 줄 수: 590


,열 이름,빈칸 비율(%),값 종류 수,표준편차,최솟값,최댓값
292,sensor_293,91.19,92,0.011494,0.0041,0.0831
293,sensor_294,91.19,138,137.692483,82.3233,879.2260
158,sensor_159,91.19,138,406.848810,234.0996,2505.2998
157,sensor_158,91.19,128,0.039538,0.0118,0.2876
492,sensor_493,85.58,226,1.759262,4.8882,21.0443
85,sensor_086,85.58,97,0.002928,0.1053,0.1184
358,sensor_359,85.58,20,0.000395,0.0017,0.0047
220,sensor_221,85.58,69,0.001989,0.0057,0.0240
244,sensor_245,64.96,66,0.084618,0.0003,1.9844
517,sensor_518,64.96,543,4.890663,0.2880,113.2758


## Step 4. 위아래를 훑어보며 기준 세우기

In [4]:
# 빈칸 비율 낮은 순으로 정렬해서 위 10줄
sensor_diagnosis.sort_values("빈칸 비율(%)", ascending=True).head(10)


,열 이름,빈칸 비율(%),값 종류 수,표준편차,최솟값,최댓값
522,sensor_523,0.0,1562,7.104435,2.6811,137.9838
527,sensor_528,0.0,1549,1.888698,2.1700,14.4479
526,sensor_527,0.0,1514,0.958428,0.1705,8.2037
524,sensor_525,0.0,1543,20.663414,1.3104,818.0005
520,sensor_521,0.0,1536,5.702366,0.3121,111.7365
523,sensor_524,0.0,1040,4.147581,0.0258,111.3330
521,sensor_522,0.0,9,103.122996,0.0000,1000.0000
493,sensor_494,0.0,572,0.973948,0.8330,9.4024
392,sensor_393,0.0,109,0.002956,0.0005,0.0229
495,sensor_496,0.0,964,3.260019,1.7720,107.6926


In [5]:
# 값 종류 수가 1개인 열(상수열) 개수
상수열_개수 = (sensor_diagnosis["값 종류 수"] == 1).sum()
print("값 종류 수가 1개인 열 개수:", 상수열_개수)

# 표준편차가 0인 열 개수
표준편차0_개수 = (sensor_diagnosis["표준편차"] == 0).sum()
print("표준편차가 0인 열 개수:", 표준편차0_개수)


값 종류 수가 1개인 열 개수: 116
표준편차가 0인 열 개수: 116


[내가 정한 1차 선별 기준]<br>
1. 빈칸 비율이 [50]% 이상인 열은 버린다<br>
2. 값 종류가 1개인 열은 버린다 (상수열)<br>
3. 표준편차가 [0.001] 이하인 열은 버린다

## Step 5. 기준대로 걸러내기

In [6]:
# 조건별로 걸리는 열 각각 확인
조건_빈칸 = sensor_diagnosis["빈칸 비율(%)"] >= 50
조건_상수 = sensor_diagnosis["값 종류 수"] == 1
조건_저변동 = sensor_diagnosis["표준편차"] <= 0.001

print("빈칸 비율 50% 이상:", 조건_빈칸.sum(), "개")
print("값 종류 수 1개:", 조건_상수.sum(), "개")
print("표준편차 0.001 이하:", 조건_저변동.sum(), "개")

# 하나라도 걸리면 제외 대상 (중복 제거)
제외_조건 = 조건_빈칸 | 조건_상수 | 조건_저변동
print("중복 제외한 총 제외 대상:", 제외_조건.sum(), "개")

# 원본 sensor_diagnosis는 그대로 두고, 걸러낸 새 표를 만든다
sensor_diagnosis_filtered = sensor_diagnosis[~제외_조건].copy()

print("남은 열 개수:", len(sensor_diagnosis_filtered))


빈칸 비율 50% 이상: 28 개
값 종류 수 1개: 116 개
표준편차 0.001 이하: 127 개
중복 제외한 총 제외 대상: 154 개
남은 열 개수: 436


[1차 선별 결과]<br>
시작 : 센서 590개<br>
- 빈칸 과다로 제외 : [28]개<br>
- 상수열로 제외 : [116]개<br>
- 거의 안 변해서 제외 : [127]개<br>
(중복 제외) 총 제외 : [154]개<br>
남은 센서 : [436]개

## Step 6. 하나만 직접 확인하기

### 상수열 하나를 직접 확인
걸러낸 열이 정말 값이 하나뿐인지 눈으로 본다.

In [7]:
# value_counts() — 어떤 값이 각각 몇 번 나오는지 세어준다
# 따옴표 안은 예시입니다. AI가 알려준 상수열 목록에서 아무거나 하나로 바꾸세요
df["sensor_014"].value_counts()

sensor_014
0.0    1564
Name: count, dtype: int64

## Step 7. 아직 몇 개가 남았나

590개에서 \[436\]개로 줄었다. 그런데 이것도 관리도를 [436]개 그려야 한다.<br>
더 줄일 방법이 있을까? : [비슷하게 움직이는 센서끼리는 하나만 봐도 될 것 같다.<br>
그리고 판정과 아무 관계없는 열은 아예 뺄 수 있지 않을까]

---
## 직접 해보기 (도전) - 기준을 하나만 바꿔보면

- 상황: 빈칸 비율 기준을 내가 정했지만, 좁히면 얼마나 달라지는지는 아직 모른다
- 할 일: 기준을 하나만 좁혀서 다시 걸러내고 숫자를 비교한다
- 결과물: 세 줄짜리 비교표 1개

In [8]:
# 상수열 / 저변동 조건은 두 기준 모두 동일하게 적용
조건_상수 = sensor_diagnosis["값 종류 수"] == 1
조건_저변동 = sensor_diagnosis["표준편차"] <= 0.001

def 빈칸기준_비교(임계값):
    조건_빈칸 = sensor_diagnosis["빈칸 비율(%)"] >= 임계값
    제외_조건 = 조건_빈칸 | 조건_상수 | 조건_저변동
    return {
        "기준": f"빈칸 비율 {임계값}% 이상 제외",
        "빈칸 과다로 제외되는 열 개수": int(조건_빈칸.sum()),
        "중복 제외한 총 제외 열 개수": int(제외_조건.sum()),
        "남는 센서 열 개수": int((~제외_조건).sum()),
    }

기준A = 빈칸기준_비교(50)
기준B = 빈칸기준_비교(30)

빈칸기준_비교표 = pd.DataFrame([기준A, 기준B])
빈칸기준_비교표


,기준,빈칸 과다로 제외되는 열 개수,중복 제외한 총 제외 열 개수,남는 센서 열 개수
0,빈칸 비율 50% 이상 제외,28,154,436
1,빈칸 비율 30% 이상 제외,32,158,432


### 빈칸 기준 A vs B

| 항목 | A (50%) | B (30%) |
|---|---|---|
| 빈칸 과다로 제외 | [28] | [32] |
| 중복 뺀 총 제외 | [154] | [158] |
| 남는 센서 열 수 | [436] | [432] |

### 인사이트

- 기준을 50% → 30%로 20%p나 좁혔는데도, 빈칸 과다로 걸린 열은 28개 → 32개(+4개)로 거의 늘지 않았고, 최종 남는 열도 436개 → 432개(-4개, 약 0.9% 감소)에 그쳤다.
- 즉 이 데이터의 빈칸 비율은 30~50% 구간에 걸쳐 있는 열이 거의 없다는 뜻이다. 앞서 본 상/하위 10개 목록에서도 빈칸 비율이 0%(하위 10개) 아니면 64.96% 이상(상위 10개)으로 갈려 있어, 중간대가 비어 있는 모습과 일치한다.
- 이번 데이터에서는 최종 제외(154~158개) 중 상수열(116개)·저변동(127개) 조건이 이미 대부분을 차지하고 있어서, 빈칸 비율 기준 자체는 30~50% 사이 어디로 잡아도 결과가 크게 달라지지 않는다.
- 다만 이건 이 데이터셋의 분포 특성일 뿐, 다른 데이터에서도 항상 그렇다고 확신할 수는 없다 — 확실하지 않음.
